In [1]:
from itertools import product
from math import exp

## Substitution Matrix

Same substitution matrix helper used in the previous question:

In [2]:
def make_substitution_matrix(alphabet, match_score, mismatch_score):
    return {
        letter1: {
            letter2: match_score if letter1 == letter2 else mismatch_score
            for letter2 in alphabet
        }
        for letter1 in alphabet
    }

In [3]:
substitution_matrix = make_substitution_matrix("ACGT", 1, -1)
substitution_matrix

{'A': {'A': 1, 'C': -1, 'G': -1, 'T': -1},
 'C': {'A': -1, 'C': 1, 'G': -1, 'T': -1},
 'G': {'A': -1, 'C': -1, 'G': 1, 'T': -1},
 'T': {'A': -1, 'C': -1, 'G': -1, 'T': 1}}

## Words

We have to extract all overlapping words of length $k$ and get each word together with its zero-based start position.

In [4]:
def get_words(sequence, word_length):
    return [
        (sequence[position:position + word_length], position)
        for position in range(len(sequence) - word_length + 1)
    ]

In [5]:
get_words("AGGACT", 3)

[('AGG', 0), ('GGA', 1), ('GAC', 2), ('ACT', 3)]

## Word Scoring
Two words of the same length are scored by summing one substitution-matrix value per position:

$$
S(u,v)=\sum_{i=1}^{k}s(u_i,v_i)
$$

In [6]:
def score_pair(sequence1, sequence2, substitution_matrix):
    if len(sequence1) != len(sequence2):
        raise ValueError("Sequences must have the same length")
    return sum(
        substitution_matrix[letter1][letter2]
        for letter1, letter2 in zip(sequence1, sequence2)
    )

In [7]:
score_pair("AGG", "AGT", substitution_matrix)

1

In [8]:
# should fail with ValueError
try:
    score_pair("AGG", "AG", substitution_matrix)
except ValueError:
    print("failed successfully")
else:
    raise AssertionError("Expected ValueError")

failed successfully


## Neighborhood Words

For every query word, we have to create all possible words of the same length, over the alphabet. A generated word becomes a neighborhood word if its score against the query word is at least the word threshold $T_w$:

$$
N(w)=\{v\in\Sigma^k \mid S(w,v)\ge T_w\}
$$

We have to store the database word as key and the matching query positions as the values. This makes the database scan simple: if a database word appears in this dictionary, it is a seed hit.

In [9]:
def get_neighborhood_words(query, word_length, substitution_matrix, word_threshold):
    neighborhood_words = {}
    alphabet = sorted(substitution_matrix)

    for query_word, query_position in get_words(query, word_length):
        for letters in product(alphabet, repeat=word_length):
            database_word = "".join(letters)
            score = score_pair(query_word, database_word, substitution_matrix)
            if score >= word_threshold:
                neighborhood_words.setdefault(database_word, []).append(
                    (query_position, query_word, score)
                )

    return neighborhood_words

In [10]:
get_neighborhood_words("AGGACT", 3, substitution_matrix, 2)

{'ACT': [(3, 'ACT', 3)],
 'AGG': [(0, 'AGG', 3)],
 'GAC': [(2, 'GAC', 3)],
 'GGA': [(1, 'GGA', 3)]}

## Extension

After a seed hits, it should be extended on the same diagonal to the left and to the right. The extension function should keep the best cumulative score and the corresponding extension length. The full extension score is the left extension score plus the seed score plus the right extension score. 

In [11]:
def get_best_extension(
    query, database_sequence, query_index, database_index, limit, step, substitution_matrix
):
    best_score = best_extension = current_score = 0

    for extension in range(limit):
        current_score += substitution_matrix[
            query[query_index + step * extension]
        ][database_sequence[database_index + step * extension]]
        if current_score > best_score:
            best_score, best_extension = current_score, extension + 1

    return best_score, best_extension


def extend_hit(
    query, database_sequence, query_position, database_position, word_length, substitution_matrix
):
    seed_query = query[query_position:query_position + word_length]
    seed_database = database_sequence[database_position:database_position + word_length]

    left_score, left_extension = get_best_extension(
        query, 
        database_sequence, 
        query_position - 1, 
        database_position - 1, 
        min(query_position, database_position), 
        -1, 
        substitution_matrix
    )
    right_score, right_extension = get_best_extension(
        query,
        database_sequence,
        query_position + word_length,
        database_position + word_length,
        min(
            len(query) - query_position - word_length,
            len(database_sequence) - database_position - word_length
        ),
        1,
        substitution_matrix
    )

    query_start = query_position - left_extension
    query_end = query_position + word_length + right_extension
    database_start = database_position - left_extension
    database_end = database_position + word_length + right_extension
    score = left_score + score_pair(seed_query, seed_database, substitution_matrix) + right_score

    while (
        query_end - query_start > 1
        and substitution_matrix[query[query_start]][database_sequence[database_start]] < 0
    ):
        score -= substitution_matrix[query[query_start]][database_sequence[database_start]]
        query_start += 1
        database_start += 1

    while (
        query_end - query_start > 1
        and substitution_matrix[query[query_end - 1]][database_sequence[database_end - 1]] < 0
    ):
        score -= substitution_matrix[
            query[query_end - 1]
        ][database_sequence[database_end - 1]]
        query_end -= 1
        database_end -= 1

    return {
        "query_start": query_start,
        "query_end": query_end,
        "database_start": database_start,
        "database_end": database_end,
        "query_alignment": query[query_start:query_end],
        "database_alignment": database_sequence[database_start:database_end],
        "score": score
    }

In [12]:
extend_hit("AGGACT", "TTTAGGACGAAA", 0, 3, 3, substitution_matrix)

{'database_alignment': 'AGGAC',
 'database_end': 8,
 'database_start': 3,
 'query_alignment': 'AGGAC',
 'query_end': 5,
 'query_start': 0,
 'score': 5}

## E-value

I used the standard raw-score form for the E-value:

$$
E = Kmn e^{-\lambda S}
$$

Here $S$ is the raw alignment score, $m$ is the query length, and $n$ is the total number of letters in the searched database. For this simplified implementation I used $K=1$ and $\lambda=1$ as defaults. 

In [13]:
def get_database_entries(database):
    if isinstance(database, dict):
        return list(database.items())
    return database


def get_database_length(database_entries):
    return sum(len(sequence) for _, sequence in database_entries)


def get_e_value(score, query_length, database_length, lambda_value=1.0, k_value=1.0):
    return k_value * query_length * database_length * exp(-lambda_value * score)

In [14]:
database_entries = [
    ("db_1", "TTTAGGACGAAA"),
    ("db_2", "CCCGTGAGT"),
    ("db_3", "TTACTT")
]
get_database_length(database_entries), get_e_value(5, 6, 27)

(27, 1.0915474138518457)

## BLAST
The main BLAST function puts these steps together. It first builds the query neighborhood words, then scans every database sequence for matching words. Every seed is extended, an E-value is calculated, and only hits with score at least the hit threshold $T_h$ are returned. Optionally, the user can also pass an E-value threshold.

In [ ]:
def blast(
    query, 
    database, 
    substitution_matrix, 
    word_length, word_threshold, 
    hit_threshold, lambda_value=1.0, k_value=1.0, e_value_threshold=None
):
    if word_length <= 0:
        raise ValueError("Word length must be greater than zero")

    hits = {}
    database_entries = get_database_entries(database)
    database_length = get_database_length(database_entries)
    neighborhood_words = get_neighborhood_words(query, word_length, substitution_matrix, word_threshold)

    for database_name, database_sequence in database_entries:
        for database_word, database_position in get_words(database_sequence, word_length):
            for query_position, _, _ in neighborhood_words.get(database_word, []):
                hit = extend_hit(
                    query,
                    database_sequence,
                    query_position,
                    database_position,
                    word_length,
                    substitution_matrix
                )
                hit["e_value"] = get_e_value(
                    hit["score"],
                    len(query),
                    database_length,
                    lambda_value,
                    k_value
                )
                if (
                    hit["score"] >= hit_threshold
                    and (e_value_threshold is None or hit["e_value"] <= e_value_threshold)
                ):
                    hit["database_name"] = database_name
                    key = (
                        database_name,
                        hit["query_start"],
                        hit["query_end"],
                        hit["database_start"],
                        hit["database_end"]
                    )
                    hits[key] = hit

    return sorted(
        hits.values(),
        key=lambda hit: (
            -hit["score"],
            hit["e_value"],
            hit["database_name"],
            hit["database_start"],
            hit["query_start"]
        )
    )

In [16]:
blast(
    "AGGACT",
    database_entries,
    substitution_matrix,
    3,
    2,
    3
)

[{'database_alignment': 'AGGAC',
  'database_end': 8,
  'database_name': 'db_1',
  'database_start': 3,
  'e_value': 1.0915474138518457,
  'query_alignment': 'AGGAC',
  'query_end': 5,
  'query_start': 0,
  'score': 5},
 {'database_alignment': 'ACT',
  'database_end': 5,
  'database_name': 'db_3',
  'database_start': 2,
  'e_value': 8.06550507559396,
  'query_alignment': 'ACT',
  'query_end': 6,
  'query_start': 3,
  'score': 3}]

## Testing Helper

In [17]:
def get_match_line(aligned1, aligned2):
    match_line = []
    for letter1, letter2 in zip(aligned1, aligned2):
        if letter1 == letter2:
            match_line.append("|")
        else:
            match_line.append(".")
    return "".join(match_line)


def display_hit(hit):
    print(f"Database sequence: {hit['database_name']}")
    print(f"Query position (0-based): {hit['query_start']} to {hit['query_end'] - 1}")
    print(f"Database position (0-based): {hit['database_start']} to {hit['database_end'] - 1}")
    print(hit["query_alignment"])
    print(get_match_line(hit["query_alignment"], hit["database_alignment"]))
    print(hit["database_alignment"])
    print(f"Score: {hit['score']}")
    print(f"E-value: {hit['e_value']:.3g}")


def run_example(name, query, database, word_length, word_threshold, hit_threshold):
    substitution_matrix = make_substitution_matrix("ACGT", 1, -1)
    hits = blast(query, database, substitution_matrix, word_length, word_threshold, hit_threshold)

    print("=" * 60)
    print(name)
    print(f"Query: {query}")
    print(
        f"word_length={word_length}, "
        f"word_threshold={word_threshold}, "
        f"hit_threshold={hit_threshold}"
    )
    print()

    if not hits:
        print("No hits found.")
        return

    for hit in hits:
        display_hit(hit)
        print()

## Test Inputs

In [18]:
examples = [
    (
        "Example 1",
        "AGGACT",
        [
            ("db_1", "TTTAGGACGAAA"),
            ("db_2", "CCCGTGAGT"),
            ("db_3", "TTACTT")
        ],
        3,
        2,
        3
    ),
    (
        "Example 2",
        "ACCGTA",
        [
            ("db_1", "TTTACCGTAAA"),
            ("db_2", "GGGACGTTT"),
            ("db_3", "CCCCCC")
        ],
        3,
        3,
        4
    ),
    (
        "Example 3",
        "ACGTAC",
        [
            ("db_1", "ACGTACGTAC"),
            ("db_2", "TTTACGTAAA"),
            ("db_3", "TGCATGCA")
        ],
        3,
        3,
        4
    ),
    (
        "Example 4",
        "ACGTACGT",
        [
            ("db_1", "ACGTACGA"),
            ("db_2", "TTTACGTTT"),
            ("db_3", "GGGGTTTT")
        ],
        3,
        3,
        20
    )
]

## Example 1

In [19]:
run_example(*examples[0])

Example 1
Query: AGGACT
word_length=3, word_threshold=2, hit_threshold=3

Database sequence: db_1
Query position (0-based): 0 to 4
Database position (0-based): 3 to 7
AGGAC
|||||
AGGAC
Score: 5
E-value: 1.09

Database sequence: db_3
Query position (0-based): 3 to 5
Database position (0-based): 2 to 4
ACT
|||
ACT
Score: 3
E-value: 8.07



## Example 2

In [20]:
run_example(*examples[1])

Example 2
Query: ACCGTA
word_length=3, word_threshold=3, hit_threshold=4

Database sequence: db_1
Query position (0-based): 0 to 5
Database position (0-based): 3 to 8
ACCGTA
||||||
ACCGTA
Score: 6
E-value: 0.387



## Example 3

In [21]:
run_example(*examples[2])

Example 3
Query: ACGTAC
word_length=3, word_threshold=3, hit_threshold=4

Database sequence: db_1
Query position (0-based): 0 to 5
Database position (0-based): 0 to 5
ACGTAC
||||||
ACGTAC
Score: 6
E-value: 0.416

Database sequence: db_1
Query position (0-based): 0 to 5
Database position (0-based): 4 to 9
ACGTAC
||||||
ACGTAC
Score: 6
E-value: 0.416

Database sequence: db_2
Query position (0-based): 0 to 4
Database position (0-based): 3 to 7
ACGTA
|||||
ACGTA
Score: 5
E-value: 1.13



## Example 4

In [22]:
run_example(*examples[3])

Example 4
Query: ACGTACGT
word_length=3, word_threshold=3, hit_threshold=20

No hits found.


## Manual Workthrough

### Example 1

We'll do the manual trace for the first example. The query words of length 3 are:

| Query word | Query position |
|--|--:|
| `AGG` | 0 |
| `GGA` | 1 |
| `GAC` | 2 |
| `ACT` | 3 |

With match score $+1$, mismatch score $-1$, and word length 3, an exact word match has score 3. A word with one mismatch has the following score:

$$
1+1-1=1
$$

Since $T_w=2$, only exact words are neighborhood words in this example. Therefore, the neighborhood dictionary contains:

| Neighborhood word | Query position | Word score |
|--|--:|--:|
| `AGG` | 0 | 3 |
| `GGA` | 1 | 3 |
| `GAC` | 2 | 3 |
| `ACT` | 3 | 3 |

The database is then scanned with the same word length. The matching seed hits are:

| Database | Database word | Database position | Query position |
|--|--|--:|--:|
| `db_1` | `AGG` | 3 | 0 |
| `db_1` | `GGA` | 4 | 1 |
| `db_1` | `GAC` | 5 | 2 |
| `db_3` | `ACT` | 2 | 3 |

There is no seed hit in `db_2`, because none of its length-3 words is one of the query neighborhood words.

Now, the first seed hit is extended. The seed `AGG` occurs at query position 0 and database position 3:

| Part | Score |
|--|--|
| Seed `AGG`/`AGG` | $1+1+1=3$ |
| Left extension | 0, because the query already starts at position 0 |

The right extension checks the following diagonal positions:

| Extension step | Query letter | Database letter | Current cumulative score |
|--:|--|--|--:|
| 1 | `A` | `A` | 1 |
| 2 | `C` | `C` | 2 |
| 3 | `T` | `G` | 1 |

The best right extension is therefore two letters long, with score 2. The total score is:

$$
0 + 3 + 2 = 5
$$

So the final extended hit is:

```text
Query position:    0 to 4
Database position: 3 to 7
AGGAC
|||||
AGGAC
Score: 5
E-value: 1.09
```

For this example, the query length is $m=6$, and the total database length is:

$$
n = 12 + 9 + 6 = 27
$$

Using $K=1$, $\lambda=1$, and score $S=5$, the E-value is:

$$
E = 1\cdot 6\cdot 27\cdot e^{-5} \approx 1.09
$$

The other two `db_1` seeds, `GGA` and `GAC`, extend to the same final coordinates. The program stores hits in a dictionary using the database name and final coordinates as the key, so this alignment is printed only once.

The seed `ACT` in `db_3` has score 3. Its left extension would start with the mismatch `G`/`A`, which has score $-1$, so the best left extension score stays 0. There is no right extension because the query seed already reaches the end of the query. The final hit is therefore:

```text
Query position:    3 to 5
Database position: 2 to 4
ACT
|||
ACT
Score: 3
E-value: 8.07
```

For this second hit, the score is lower, so the E-value is higher:

$$
E = 1\cdot 6\cdot 27\cdot e^{-3} \approx 8.07
$$

Both hits pass the hit threshold $T_h=3$. This gives exactly the two hits printed in the first terminal example. The `db_1` hit is more significant because it has the lower E-value.

### Example 3

The third example checks that the same query can match the same database sequence at more than one position. The query is `ACGTAC` and `db_1` is `ACGTACGTAC`. The exact query occurs at database positions 0 and 4:

| Database | Query range | Database range | Score | E-value |
|--|--:|--:|--:|--:|
| `db_1` | 0-5 | 0-5 | 6 | 0.416 |
| `db_1` | 0-5 | 4-9 | 6 | 0.416 |
| `db_2` | 0-4 | 3-7 | 5 | 1.13 |

The two `db_1` hits are both kept because their database positions are different. This is why they have different dictionary keys. They also have the same E-value, because they have the same score and are evaluated against the same query and database size.

The fourth test case checks the threshold behavior. The query is `ACGTACGT` and the hit threshold is set to 20. With match score $+1$, even a perfect alignment of all eight query letters can only score:

$$
8\cdot 1 = 8
$$

Since $8 < 20$, no possible hit can pass the threshold. The correct output is therefore:

```text
No hits found.
```